In [42]:
import cv2
import mediapipe as mp
import numpy as np
from collections import deque
import pandas as pd

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils


def calculate_angle(point_a, point_b, point_c):
    point_a, point_b, point_c = np.array(point_a), np.array(point_b), np.array(point_c)
    vector_ba = point_a - point_b
    vector_bc = point_c - point_b
    cosine_angle = np.dot(vector_ba, vector_bc) / (np.linalg.norm(vector_ba) * np.linalg.norm(vector_bc))
    angle_in_degrees = np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
    return angle_in_degrees


video_path = './Videos/fr2.mp4'
video_capture = cv2.VideoCapture(video_path)
frames_per_second = video_capture.get(cv2.CAP_PROP_FPS)

# --- iste kontrole kao u side view ---
push_off_threshold = 0.05
min_swing_frames = frames_per_second * 0.22
min_contact_frames = 5

all_step_metrics_storage = []

# STRIKE detekcija preko ankle_y lokalnog maksimuma (front view)
history_window_size = 9
right_ankle_y_history = deque(maxlen=history_window_size)
left_ankle_y_history = deque(maxlen=history_window_size)

leg_is_on_ground = {"Right": False, "Left": False}
ankle_y_at_contact = {"Right": None, "Left": None}

last_leg_that_landed = None
last_strike_frame_index = 0
previous_strike_frame = None
current_instant_cadence = 0

right_step_count = 0
left_step_count = 0
current_status_event = "CALIBRATING"


with mp_pose.Pose(min_detection_confidence=0.7, min_tracking_confidence=0.7) as pose_analyzer:
    while video_capture.isOpened():
        ret, frame = video_capture.read()
        if not ret:
            break

        overlay_layer = frame.copy()
        display_frame = frame.copy()
        frame_height, frame_width, _ = frame.shape
        results = pose_analyzer.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            def get_pixel_point(idx):
                return np.array([
                    int(landmarks[idx].x * frame_width),
                    int(landmarks[idx].y * frame_height)
                ])

            # ---------- PIXEL POINTS (za crtanje/overlay) ----------
            right_hip_px, right_knee_px, right_ankle_px = get_pixel_point(24), get_pixel_point(26), get_pixel_point(28)
            left_hip_px, left_knee_px, left_ankle_px = get_pixel_point(23), get_pixel_point(25), get_pixel_point(27)

            right_shoulder_px, right_elbow_px, right_wrist_px = get_pixel_point(12), get_pixel_point(14), get_pixel_point(16)
            left_shoulder_px, left_elbow_px, left_wrist_px = get_pixel_point(11), get_pixel_point(13), get_pixel_point(15)

            nose_px = get_pixel_point(0)

            # ---------- NORMALIZED POINTS (za računanje metrika; stabilnije) ----------
            L_hip = np.array([landmarks[23].x, landmarks[23].y])
            R_hip = np.array([landmarks[24].x, landmarks[24].y])

            L_sh = np.array([landmarks[11].x, landmarks[11].y])
            R_sh = np.array([landmarks[12].x, landmarks[12].y])

            L_ank = np.array([landmarks[27].x, landmarks[27].y])
            R_ank = np.array([landmarks[28].x, landmarks[28].y])

            L_wri = np.array([landmarks[15].x, landmarks[15].y])
            R_wri = np.array([landmarks[16].x, landmarks[16].y])

            nose = np.array([landmarks[0].x, landmarks[0].y])

            # ---------- SCALE REFERENCES ----------
            torso_height = abs(((L_hip[1] + R_hip[1]) / 2) - ((L_sh[1] + R_sh[1]) / 2))
            mid_hip_x = (L_hip[0] + R_hip[0]) / 2
            hip_width = abs(R_hip[0] - L_hip[0])

            # ---------- FRONT VIEW METRICS ----------
            # HIP DROP (0–90°)
            dx_hip = abs(R_hip[0] - L_hip[0])
            dy_hip = abs(R_hip[1] - L_hip[1])
            hip_drop_value_deg = np.degrees(np.arctan2(dy_hip, dx_hip + 1e-9))

            # SHOULDER DROP (0–90°)
            dx_sh = abs(R_sh[0] - L_sh[0])
            dy_sh = abs(R_sh[1] - L_sh[1])
            shoulder_drop_value_deg = np.degrees(np.arctan2(dy_sh, dx_sh + 1e-9))

            # FOOT OFFSET (RELATIVE)
            right_foot_offset_rel = (R_ank[0] - mid_hip_x) / (hip_width + 1e-9)
            left_foot_offset_rel = (L_ank[0] - mid_hip_x) / (hip_width + 1e-9)

            # ARM INWARD (deg): shoulder -> wrist, kao "koliko je vodoravno" (0 vertikalno, 90 horizontalno)
            dx_r = abs(landmarks[16].x - landmarks[12].x)
            dy_r = abs(landmarks[16].y - landmarks[12].y)
            right_arm_inward_deg = np.degrees(np.arctan2(dx_r, dy_r + 1e-9))

            dx_l = abs(landmarks[15].x - landmarks[11].x)
            dy_l = abs(landmarks[15].y - landmarks[11].y)
            left_arm_inward_deg = np.degrees(np.arctan2(dx_l, dy_l + 1e-9))

            arm_inward_deg = (right_arm_inward_deg + left_arm_inward_deg) / 2

            # FIST HEIGHT (RELATIVE) u odnosu na nose
            right_fist_height_rel = (nose[1] - R_wri[1]) / (torso_height + 1e-9)
            left_fist_height_rel = (nose[1] - L_wri[1]) / (torso_height + 1e-9)

            # --- vizualizacija ---
            cv2.line(display_frame, tuple(left_hip_px), tuple(right_hip_px), (255, 255, 255), 2)
            cv2.line(display_frame, tuple(left_shoulder_px), tuple(right_shoulder_px), (255, 255, 255), 2)

            midline_x_px = int(mid_hip_x * frame_width)
            cv2.line(display_frame, (midline_x_px, 0), (midline_x_px, frame_height), (200, 200, 200), 1)

            mid_hip_y_px = int(((landmarks[23].y + landmarks[24].y) / 2) * frame_height)
            cv2.line(display_frame, tuple(nose_px), (midline_x_px, mid_hip_y_px), (200, 200, 200), 1)

            cv2.circle(display_frame, tuple(left_ankle_px), 6, (255, 0, 255), -1)
            cv2.circle(display_frame, tuple(right_ankle_px), 6, (255, 0, 255), -1)
            cv2.circle(display_frame, tuple(left_wrist_px), 6, (255, 165, 0), -1)
            cv2.circle(display_frame, tuple(right_wrist_px), 6, (255, 165, 0), -1)
            cv2.circle(display_frame, tuple(nose_px), 5, (0, 255, 255), -1)
            # --- Torso overlay kao u PolinaGitHub ---
            torso_points = np.array(
                [right_shoulder_px, left_shoulder_px, left_hip_px, right_hip_px],
                np.int32
            )
            
            cv2.fillPoly(overlay_layer, [torso_points], (0, 255, 0))
            cv2.addWeighted(overlay_layer, 0.15, display_frame, 0.85, 0, display_frame)
            
            cv2.polylines(display_frame, [torso_points], True, (255, 255, 255), 2)
            cv2.line(display_frame, tuple(right_shoulder_px), tuple(left_hip_px), (255, 255, 255), 1)
            cv2.line(display_frame, tuple(left_shoulder_px), tuple(right_hip_px), (255, 255, 255), 1)
            
            # opcionalno: linija između gležnjeva (kao side-view “base line”)
            
            #cv2.line(display_frame, tuple(right_hip), (right_hip[0], right_hip[1]-80), (0, 0, 255), 1)

            # --- noge (hip->knee->ankle) ---
            for h, k, a, col in [
                (right_hip_px, right_knee_px, right_ankle_px, (0, 255, 0)),
                (left_hip_px,  left_knee_px,  left_ankle_px,  (0, 255, 255))
            ]:
                cv2.line(display_frame, tuple(h), tuple(k), col, 3)
                cv2.line(display_frame, tuple(k), tuple(a), col, 3)
            
            # --- ruke (shoulder->elbow->wrist) ---
            for s, e, w in [
                (right_shoulder_px, right_elbow_px, right_wrist_px),
                (left_shoulder_px,  left_elbow_px,  left_wrist_px)
            ]:
                cv2.line(display_frame, tuple(s), tuple(e), (255, 165, 0), 3)
                cv2.line(display_frame, tuple(e), tuple(w), (255, 165, 0), 3)

            # Midline to ankle arrows (foot offset vizualno)
            midline_pt_r = (midline_x_px, int(R_ank[1] * frame_height))
            midline_pt_l = (midline_x_px, int(L_ank[1] * frame_height))
            
            cv2.arrowedLine(display_frame, midline_pt_r, tuple(right_ankle_px), (255, 0, 255), 2, tipLength=0.2)
            cv2.arrowedLine(display_frame, midline_pt_l, tuple(left_ankle_px),  (255, 0, 255), 2, tipLength=0.2)
            
            # label abs vrijednost (da vidiš "koliko", bez znaka)
            cv2.putText(display_frame, f"{abs(right_foot_offset_rel):.3f}", tuple(right_ankle_px + [8, -8]), 1, 1.0, (255, 0, 255), 1)
            cv2.putText(display_frame, f"{abs(left_foot_offset_rel):.3f}",  tuple(left_ankle_px + [8, -8]),  1, 1.0, (255, 0, 255), 1)

            # Fist height lines (wrist -> nose y)
            nose_y_px = int(nose[1] * frame_height)
            
            # Right
            cv2.line(display_frame, tuple(right_wrist_px), (right_wrist_px[0], nose_y_px), (0, 255, 255), 2)
            # Left
            cv2.line(display_frame, tuple(left_wrist_px), (left_wrist_px[0], nose_y_px), (0, 255, 255), 2)

            # HUD
            cv2.rectangle(display_frame, (0, 0), (420, 220), (20, 20, 20), -1)
            cv2.putText(display_frame, f"Hip drop: {hip_drop_value_deg:.1f} deg", (15, 30), 1, 1.0, (255, 255, 255), 1)
            cv2.putText(display_frame, f"Shoulder drop: {shoulder_drop_value_deg:.1f} deg", (15, 55), 1, 1.0, (255, 255, 255), 1)
            cv2.putText(display_frame, f"Foot offset R/L: {right_foot_offset_rel:+.2f} / {left_foot_offset_rel:+.2f}", (15, 80), 1, 1.0, (255, 255, 255), 1)
            cv2.putText(display_frame, f"Arm inward R/L: {right_arm_inward_deg:.1f} / {left_arm_inward_deg:.1f} deg", (15, 105), 1, 1.0, (255, 255, 255), 1)
            cv2.putText(display_frame, f"Fist height R/L: {right_fist_height_rel:+.2f} / {left_fist_height_rel:+.2f}", (15, 130), 1, 1.0, (255, 255, 255), 1)
            cv2.putText(display_frame, f"Steps: {right_step_count + left_step_count}", (15, 155), 1, 1.0, (0, 255, 0), 2)
            cv2.putText(display_frame, f"Cadence: {current_instant_cadence:.1f} spm", (15, 180), 1, 1.0, (0, 255, 255), 1)
            cv2.putText(display_frame, f"Status: {current_status_event}", (15, 205), 1, 1.0, (0, 255, 255), 1)

            # --- ankle_y history (za STRIKE detekciju) ---
            right_ankle_y_history.append(landmarks[28].y)
            left_ankle_y_history.append(landmarks[27].y)

            current_frame = int(video_capture.get(cv2.CAP_PROP_POS_FRAMES))

            # ---------- STRIKE (isti cadence flow kao side view) ----------
            if len(right_ankle_y_history) == history_window_size and len(left_ankle_y_history) == history_window_size:
                middle_index = history_window_size // 2
            
                for side, history, ankle_y in [
                    ("Right", right_ankle_y_history, landmarks[28].y),
                    ("Left",  left_ankle_y_history,  landmarks[27].y)
                ]:
                    if (
                        last_leg_that_landed != side and
                        history[middle_index] == max(history) and
                        (current_frame - last_strike_frame_index) > min_swing_frames
                    ):
                        # --- STRIKE ---
                        leg_is_on_ground[side] = True
                        ankle_y_at_contact[side] = ankle_y
                        last_leg_that_landed = side
                        current_status_event = f"{side.upper()} STRIKE"
            
                        # --- CADENCE (identično side view) ---
                        if previous_strike_frame is not None:
                            frames_between = current_frame - previous_strike_frame
                            if frames_between > 0:
                                current_instant_cadence = (60 * frames_per_second) / frames_between
            
                        previous_strike_frame = current_frame
                        last_strike_frame_index = current_frame
            
                        # step count
                        if side == "Right":
                            right_step_count += 1
                        else:
                            left_step_count += 1
            
                        # side-specific metrike u ovom frameu
                        foot_now = right_foot_offset_rel if side == "Right" else left_foot_offset_rel
                        fist_now = right_fist_height_rel if side == "Right" else left_fist_height_rel
            
                        # record (start kontakt)
                        all_step_metrics_storage.append({
                            "side": side,
                            "start_frame": current_frame,
                            "cadence": current_instant_cadence,
                            "done": False,
            
                            "hip_drop_max_deg": hip_drop_value_deg,
                            "shoulder_drop_max_deg": shoulder_drop_value_deg,
            
                            # mean akumulatori (kroz kontakt)
                            "arm_inward_sum": arm_inward_deg,
                            "fist_height_sum": fist_now,
                            "foot_offset_sum": foot_now,
                            "n": 1
                        })
            
            
            # ---------- PUSH-OFF + GCT (identično mjerenje kao side view) ----------
            for side, current_y in [("Right", landmarks[28].y), ("Left", landmarks[27].y)]:
                if leg_is_on_ground[side]:
                    for record in reversed(all_step_metrics_storage):
                        if record["side"] == side and not record["done"]:
            
                            # update max drop
                            if hip_drop_value_deg > record["hip_drop_max_deg"]:
                                record["hip_drop_max_deg"] = hip_drop_value_deg
                            if shoulder_drop_value_deg > record["shoulder_drop_max_deg"]:
                                record["shoulder_drop_max_deg"] = shoulder_drop_value_deg
            
                            # update mean akumulatore
                            record["arm_inward_sum"] += arm_inward_deg
                            record["fist_height_sum"] += (right_fist_height_rel if side == "Right" else left_fist_height_rel)
                            record["foot_offset_sum"] += (right_foot_offset_rel if side == "Right" else left_foot_offset_rel)
                            record["n"] += 1
            
                            frames_on_ground = current_frame - record["start_frame"]
            
                            # push-off kad se gležanj dovoljno "podigne"
                            if (ankle_y_at_contact[side] - current_y) > push_off_threshold and frames_on_ground >= min_contact_frames:
                                leg_is_on_ground[side] = False
                                current_status_event = f"{side.upper()} PUSH-OFF"
            
                                # GCT u ms
                                record["gct"] = (frames_on_ground / frames_per_second) * 1000
            
                                n = max(1, record["n"])
                                record["arm_inward_mean_deg"] = record["arm_inward_sum"] / n
                                record["fist_height_mean_rel"] = record["fist_height_sum"] / n
                                record["foot_offset_mean_rel"] = record["foot_offset_sum"] / n
            
                                # cleanup
                                del record["arm_inward_sum"]
                                del record["fist_height_sum"]
                                del record["foot_offset_sum"]
                                del record["n"]
            
                                record["done"] = True
            
                            break


        cv2.imshow("Running analysis", display_frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

print("Opened:", video_capture.isOpened())
print("FPS:", frames_per_second)
print("TOTAL STEPS:", right_step_count + left_step_count)
print("DONE STEPS:", sum(1 for s in all_step_metrics_storage if s.get("done")))

# ispis svega što je izračunao (samo done koraci)
for i, s in enumerate([x for x in all_step_metrics_storage if x.get("done")], start=1):
    print(f"\n--- STEP {i} ---")
    for k, v in s.items():
        print(f"{k}: {v}")

video_capture.release()
cv2.destroyAllWindows()


I0000 00:00:1769185441.872605  101104 gl_context.cc:357] GL version: 2.1 (2.1 Metal - 90.5), renderer: Apple M2
W0000 00:00:1769185441.972825  151001 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1769185441.986972  151001 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
/opt/miniconda3/envs/runai/lib/python3.11/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Opened: True
FPS: 60.028554809370206
TOTAL STEPS: 83
DONE STEPS: 82

--- STEP 1 ---
side: Right
start_frame: 25
cadence: 0
done: True
hip_drop_max_deg: 4.0621460590133305
shoulder_drop_max_deg: 1.8648585119881935
gct: 399.8097251585624
arm_inward_mean_deg: 11.649642620691736
fist_height_mean_rel: -0.9425089551235488
foot_offset_mean_rel: -0.1254083299935897

--- STEP 2 ---
side: Left
start_frame: 59
cadence: 105.93274378124154
done: True
hip_drop_max_deg: 1.8324614285320968
shoulder_drop_max_deg: 2.4531750429868975
gct: 199.9048625792812
arm_inward_mean_deg: 13.415375719688138
fist_height_mean_rel: -0.8236858512585393
foot_offset_mean_rel: 0.31822672615601

--- STEP 3 ---
side: Right
start_frame: 80
cadence: 171.51015659820058
done: True
hip_drop_max_deg: 3.49439839905543
shoulder_drop_max_deg: 2.7048402099003463
gct: 199.9048625792812
arm_inward_mean_deg: 7.364813342941767
fist_height_mean_rel: -0.9853788129579032
foot_offset_mean_rel: -0.054750970476040384

--- STEP 4 ---
side: Left


In [38]:
print("TOTAL STEPS:", len(all_step_metrics_storage))
print("DONE STEPS:", sum(1 for s in all_step_metrics_storage if s.get("done")))


# kreiranje tablice za FRONT VIEW dataset
import csv
import os

file_path = 'dataset_frontview.csv'

fieldnames = [
    'rep_number',
    'frame_index',
    'file_name',
    'side',

    # EVENT METRICS
    'cadence_value_spm',
    'gct_value_ms',

    # FRONT VIEW METRICS (VALUES)
    'hip_drop_max_deg',
    'shoulder_drop_max_deg',
    'foot_offset_mean_rel',
    'arm_inward_mean_deg',
    'fist_height_mean_rel',

    # SCORES (prema tablici)
    'cadence_score',
    'gct_score',
    'hip_drop_score',
    'shoulder_drop_score',
    'foot_offset_score',
    'arm_inward_score',
    'fist_height_score'
]

with open(file_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
print(f"Tablica {file_path} resetirana.")


TOTAL STEPS: 83
DONE STEPS: 82
Tablica dataset_frontview.csv resetirana.


In [39]:
# dodavanje u FRONT VIEW dataset
import csv
import os

file_path = "dataset_frontview_GCT.csv"

# CSV stupci (MORA se poklapati s row ključevima)
fieldnames = [
    "rep_number", "frame_index", "file_name", "side",
    "cadence_value_spm", "gct_value_ms",
    "hip_drop_max_deg", "shoulder_drop_max_deg",
    "foot_offset_mean_rel", "arm_inward_mean_deg", "fist_height_mean_rel",
    "cadence_score", "gct_score",
    "hip_drop_score", "shoulder_drop_score",
    "foot_offset_score", "arm_inward_score", "fist_height_score"
]

print("CSV absolute path:", os.path.abspath(file_path))

# OVERWRITE + HEADER
with open(file_path, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()

print("CSV resetiran (overwrite) i header upisan.")


def get_metric_score_front(metric, val):
    if metric == "cadence":
        if val > 175: return 3
        if val > 168: return 2
        return 1

    if metric == "gct":
        if val < 265: return 3
        if val < 283: return 2
        return 1

    if metric == "hip_drop":
        if val < 5: return 3
        if val < 7: return 2
        if val < 10: return 1
        return 0

    if metric == "shoulder_drop":
        if val < 5: return 3
        if val < 7: return 2
        if val < 10: return 1
        return 0

    if metric == "foot_offset_mean_rel":
        a = abs(val)
        if a <= 0.12: return 3
        if a <= 0.20: return 2
        if a <= 0.28: return 1
        return 0

    if metric == "arm_inward_deg":
        if val < 15: return 3
        if val < 30: return 2
        if val < 45: return 1
        return 0

    if metric == "fist_height_rel":
        if val <= 0.10: return 3
        if val <= 0.20: return 2
        if val <= 0.30: return 1
        return 0

    return 0


written = 0

if "all_step_metrics_storage" in locals() and all_step_metrics_storage:
    with open(file_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")

        for i, m in enumerate(all_step_metrics_storage, start=1):
            if not m.get("done"):
                continue

            cad = float(m.get("cadence", 0))
            gct = float(m.get("gct", 0))

            hip_drop = float(m.get("hip_drop_max_deg", 0))
            sh_drop  = float(m.get("shoulder_drop_max_deg", 0))

            foot_off = float(m.get("foot_offset_mean_rel", 0))
            arm_in   = float(m.get("arm_inward_mean_deg", 0))
            fist_h   = float(m.get("fist_height_mean_rel", 0))

            row = {
                "rep_number": i,
                "frame_index": int(m.get("start_frame", 0)),
                "file_name": video_path,
                "side": m.get("side", ""),

                "cadence_value_spm": round(cad, 2),
                "gct_value_ms": int(gct),

                "hip_drop_max_deg": round(hip_drop, 2),
                "shoulder_drop_max_deg": round(sh_drop, 2),

                "foot_offset_mean_rel": round(foot_off, 4),
                "arm_inward_mean_deg": round(arm_in, 2),
                "fist_height_mean_rel": round(fist_h, 4),

                "cadence_score": get_metric_score_front("cadence", cad),
                "gct_score": get_metric_score_front("gct", gct),
                "hip_drop_score": get_metric_score_front("hip_drop", hip_drop),
                "shoulder_drop_score": get_metric_score_front("shoulder_drop", sh_drop),
                "foot_offset_score": get_metric_score_front("foot_offset_mean_rel", foot_off),
                "arm_inward_score": get_metric_score_front("arm_inward_deg", arm_in),
                "fist_height_score": get_metric_score_front("fist_height_rel", fist_h),
            }

            writer.writerow(row)
            written += 1

    print(f"Upisano redova: {written}")
    print("CSV size (bytes):", os.path.getsize(file_path))
else:
    print("all_step_metrics_storage je prazan ili ne postoji.")


CSV absolute path: /Users/nola.bonacic/Downloads/running_app-main/data_collection/dataset_frontview_GCT.csv
CSV resetiran (overwrite) i header upisan.
Upisano redova: 82
CSV size (bytes): 7325


In [41]:
# visualization (FRONT VIEW)
if all_step_metrics_storage:
    final_results = []

    keys = [
        "cadence",
        "gct",
        "hip_drop_max_deg",
        "shoulder_drop_max_deg",
        "foot_offset_mean_rel",
        "arm_inward_mean_deg",
        "fist_height_mean_rel"
    ]

    score_metric_map = {
        "cadence": "cadence",
        "gct": "gct",
        "hip_drop_max_deg": "hip_drop",
        "shoulder_drop_max_deg": "shoulder_drop",
        "foot_offset_mean_rel": "foot_offset_mean_rel",
        "arm_inward_mean_deg": "arm_inward_deg",
        "fist_height_mean_rel": "fist_height_rel"
    }

    for k in keys:
        valid_vals = [
            step[k] for step in all_step_metrics_storage
            if step.get("done") is True and (k in step) and (step[k] is not None)
        ]

        # foot offset prosjek kao mean(abs)
        if k == "foot_offset_mean_rel":
            avg = float(np.mean([abs(v) for v in valid_vals])) if valid_vals else 0.0
        else:
            avg = float(np.mean(valid_vals)) if valid_vals else 0.0

        score = get_metric_score_front(score_metric_map[k], avg)

        # ispis: foot offset bez znaka, ostalo ostaje signed
        avg_to_print = abs(avg) if k == "foot_offset_mean_rel" else avg

        # ✅ APPEND MORA BITI UNUTAR FOR PETLJE
        final_results.append({
            "METRIC": k.upper().replace("_", " "),
            "AVG VALUE": round(avg_to_print, 3) if "rel" in k else round(avg_to_print, 1),
            "SCORE": score
        })

    df = pd.DataFrame(final_results)

    print("\n" + "═" * 55)
    print("         FRONT VIEW GAIT PERFORMANCE SUMMARY")
    print("═" * 55)
    print(df.to_string(index=False))
    print("─" * 55)
    print(f"TOTAL SCORE: {df['SCORE'].sum()} / {len(keys) * 3}")
    print("═" * 55)

else:
    print("No complete gait data captured.")




═══════════════════════════════════════════════════════
         FRONT VIEW GAIT PERFORMANCE SUMMARY
═══════════════════════════════════════════════════════
               METRIC  AVG VALUE  SCORE
              CADENCE    160.000      1
                  GCT    156.000      3
     HIP DROP MAX DEG      3.300      3
SHOULDER DROP MAX DEG      2.700      3
 FOOT OFFSET MEAN REL      0.186      2
  ARM INWARD MEAN DEG     10.600      3
 FIST HEIGHT MEAN REL     -0.945      3
───────────────────────────────────────────────────────
TOTAL SCORE: 18 / 21
═══════════════════════════════════════════════════════
